# Model: learn correlations

## Part 0 — Image similarity metrics

Before the model, we define how to measure how close a predicted image is to the target (edited) image. Three metrics:

- **`relative_mae`** — your metric: sum of absolute grayscale differences divided by the sum of the target's grayscale values. Lower = better (0 = identical). Good for overall brightness deviation.
- **`psnr`** — peak signal-to-noise ratio (dB). Higher = better. Standard pixel-level reconstruction metric.
- **`ssim`** — structural similarity (−1…1, 1 = identical). Higher = better. Captures local structure/contrast, i.e. whether the images *look* alike.

All metrics operate on **grayscale** versions (RGB is converted with the standard luminance formula via `Image.convert('L')`).

In [ ]:
import os
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")


def load_gray(path):
    """Load an image as a float32 grayscale array in [0, 255]."""
    with Image.open(path) as im:
        return np.asarray(im.convert("L"), dtype=np.float32)

In [ ]:
def relative_mae(pred, target, eps=1e-8):
    """Your metric: sum|pred-target| / sum(target). Lower = better (0 = identical).
    eps guards against an all-black target."""
    pred = np.asarray(pred, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    return float(np.sum(np.abs(pred - target)) / (np.sum(target) + eps))


def mae(pred, target):
    """Plain mean absolute error (per pixel). Lower = better."""
    return float(np.mean(np.abs(np.asarray(pred, np.float32) - np.asarray(target, np.float32))))


def psnr(pred, target, data_range=255.0):
    """Peak signal-to-noise ratio in dB. Higher = better."""
    pred = np.asarray(pred, np.float32)
    target = np.asarray(target, np.float32)
    mse = np.mean((pred - target) ** 2)
    if mse == 0:
        return float("inf")
    return float(20 * np.log10(data_range) - 10 * np.log10(mse))


def ssim(pred, target, data_range=255.0):
    """Structural similarity, -1..1. Higher = better."""
    return float(structural_similarity(
        np.asarray(target, np.float32), np.asarray(pred, np.float32), data_range=data_range))


def compare(pred_path, target_path):
    """All metrics for one image pair (paths)."""
    p, t = load_gray(pred_path), load_gray(target_path)
    return {
        "relative_mae": relative_mae(p, t),
        "mae": mae(p, t),
        "psnr": psnr(p, t),
        "ssim": ssim(p, t),
    }

In [ ]:
def evaluate_folders(pred_dir, target_dir):
    """Average each metric over all filenames present in both folders."""
    p = set(f for f in os.listdir(pred_dir) if f.lower().endswith(".jpg"))
    t = set(f for f in os.listdir(target_dir) if f.lower().endswith(".jpg"))
    common = sorted(p & t)
    if not common:
        print("no common files")
        return {}
    acc = {"relative_mae": 0.0, "mae": 0.0, "psnr": 0.0, "ssim": 0.0}
    for f in common:
        m = compare(os.path.join(pred_dir, f), os.path.join(target_dir, f))
        for k in acc:
            acc[k] += m[k]
    avg = {k: v / len(common) for k, v in acc.items()}
    print(f"compared {len(common)} pairs")
    for k, v in avg.items():
        print(f"  {k}: {v:.4f}")
    return avg

In [ ]:
# Baseline sanity check: how different are original vs edited on the test set?
# This is the gap the model must close (and the floor any model should beat).
test_orig = os.path.join(DATA_DIR, "720*480", "test", "original")
test_edit = os.path.join(DATA_DIR, "720*480", "test", "edited")
print("original -> edited (identity baseline):")
evaluate_folders(test_orig, test_edit)